# BIS SIH26107 — Phase 4 Qdrant GPU Embedding

**Purpose**: Embed all 45,523 BIS knowledge chunks with BGE-M3 and upsert them to Qdrant Cloud `bis_knowledge`.

## Before running:
1. `Runtime → Change runtime type → T4 GPU`
2. Upload `qdrant_chunks.jsonl` to Google Drive at `MyDrive/BIS_SIH26107/qdrant_chunks.jsonl`
3. Run cells **top to bottom** — do not skip any cell.

> **Resumable**: if Colab disconnects, re-run Cells 1–8 to reload state, then re-run Cell 10. Already-upserted shards are automatically skipped.

## Cell 1 — Verify GPU

In [1]:
# ============================================================
# CELL 1: Verify GPU is active
# ============================================================
import subprocess

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'nvidia-smi not found')

import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name      :', torch.cuda.get_device_name(0))
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print('VRAM (GB)     :', round(vram, 1))
else:
    raise RuntimeError(
        '\u274c No GPU found. Go to Runtime \u2192 Change runtime type \u2192 T4 GPU'
    )

Thu Sep  3 15:05:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Cell 2 — Mount Drive & Install Dependencies

In [2]:
# ============================================================
# CELL 2: Mount Google Drive + install packages
# sentence-transformers is NOT installed (incompatible with this repo)
# ============================================================
import os, subprocess
from google.colab import drive

drive.mount('/content/drive')

FILE_PATH = '/content/drive/MyDrive/BIS_SIH26107/qdrant_chunks.jsonl'
if not os.path.exists(FILE_PATH):
    raise FileNotFoundError(
        f'\u274c File not found: {FILE_PATH}\n'
        'Upload qdrant_chunks.jsonl to MyDrive/BIS_SIH26107/ first.'
    )
size_mb = os.path.getsize(FILE_PATH) / 1024**2
print(f'\u2705 File found: {FILE_PATH}  ({size_mb:.1f} MB)')

print('Installing dependencies ...')
subprocess.run(['pip', 'install', '-q',
                'transformers>=4.44',
                'qdrant-client>=1.12',
                'numpy>=1.26'], check=True)
print('\u2705 Dependencies ready')

Mounted at /content/drive
✅ File found: /content/drive/MyDrive/BIS_SIH26107/qdrant_chunks.jsonl  (47.5 MB)
Installing dependencies ...
✅ Dependencies ready


## Cell 3 — Configuration (Exact Match to phase4/config.py)

In [3]:
# ============================================================
# CELL 3: Constants — must exactly match phase4/config.py
#         and phase4/qdrant/embed_and_upsert.py
#         DO NOT change any of these values
# ============================================================

# ---- Qdrant Cloud credentials ----
QDRANT_URL     = 'https://19f1b5d7-1daa-4e9f-9e80-5c1da6f2e998.eu-west-1-0.aws.cloud.qdrant.io'
QDRANT_API_KEY = ('eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9'
                  '.eyJhY2Nlc3MiOiJtIiwic3ViamVjdCI6ImFwaS1rZXk6'
                  'NjEyMDMwMjgtZmRlYy00YmY3LThmNjctOTQ3ODIyMzRlYTU5In0'
                  '.iue4V8sLMmEIZ4ARHDjgC3BKnDPNN9AJB623cZDhXGQ')
COLLECTION     = 'bis_knowledge'

# ---- Data path ----
FILE_PATH = '/content/drive/MyDrive/BIS_SIH26107/qdrant_chunks.jsonl'

# ---- phase4/config.py constants (DO NOT change) ----
EMBEDDING_MODEL    = 'BAAI/bge-m3'
POINT_ID_NAMESPACE = 'bis-sih26107-phase4/'   # config.py line 94
EXPECTED_CHUNKS    = 45523                     # config.py line 79
VECTOR_SIZE        = 1024                      # BGE-M3 dense dim
EMBEDDING_BATCH    = 64                        # config.py line 87
EMBEDDING_SHARD    = 512                       # config.py line 88
UPSERT_BATCH       = 256                       # config.py line 89
MAX_SEQ            = 1024                      # config.py line 86

# ---- Payload fields (embed_and_upsert.py lines 30-33) ----
PAYLOAD_FIELDS = (
    'chunk_id', 'document_id', 'is_id', 'canonical_is_number',
    'document_type', 'title', 'section', 'clause', 'page_start',
    'page_end', 'breadcrumb', 'content', 'source_url', 'version',
    'effective_date', 'sha256'
)
INT_FIELDS = ('page_start', 'page_end')  # embed_and_upsert.py line 29

print('\u2705 Configuration loaded')
print(f'   Model      : {EMBEDDING_MODEL}')
print(f'   Collection : {COLLECTION}')
print(f'   Expected   : {EXPECTED_CHUNKS} chunks')
print(f'   Shard size : {EMBEDDING_SHARD} | Batch: {EMBEDDING_BATCH} | Upsert: {UPSERT_BATCH}')

✅ Configuration loaded
   Model      : BAAI/bge-m3
   Collection : bis_knowledge
   Expected   : 45523 chunks
   Shard size : 512 | Batch: 64 | Upsert: 256


## Cell 4 — Load BGE-M3 on GPU

In [4]:
# ============================================================
# CELL 4: Load BAAI/bge-m3 using raw transformers.AutoModel
#         Identical to phase4/retrieval/qdrant_retriever.py
#         load_model() — lines 18-36
#         sentence-transformers is NOT used (repo intentionally avoids it)
# ============================================================
import torch
from transformers import AutoModel, AutoTokenizer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Loading {EMBEDDING_MODEL} on {device} ...')
print('First run downloads ~2.3 GB — takes 3-5 min. Subsequent runs are instant.')

tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL)
model     = AutoModel.from_pretrained(EMBEDDING_MODEL)
model.eval().to(device)

print(f'\n\u2705 Model loaded')
print(f'   Device : {device}')
print(f'   Dtype  : {next(model.parameters()).dtype}')

Loading BAAI/bge-m3 on cuda ...
First run downloads ~2.3 GB — takes 3-5 min. Subsequent runs are instant.


config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

...

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.27GB            

model.safetensors: downloading bytes:           |  0.00B            


✅ Model loaded
   Device : cuda
   Dtype  : torch.float32


## Cell 5 — Define `embed_texts()` — Exact Replica of Phase 4

In [5]:
# ============================================================
# CELL 5: embed_texts() — exact copy of
#         phase4/retrieval/qdrant_retriever.py  lines 39-54
#
#   Method: AutoModel -> CLS token (index 0) -> L2 normalize
#   Output: float32 numpy array, shape (N, 1024)
#
#   DO NOT replace with sentence-transformers — different pooling,
#   different vectors, retrieval breaks.
# ============================================================
import numpy as np
import torch

def embed_texts(texts):
    """
    Exact replica of phase4/retrieval/qdrant_retriever.py embed_texts().
    AutoTokenizer + AutoModel, CLS pooling, L2 normalize -> 1024-dim float32.
    """
    out = []
    with torch.no_grad():
        for i in range(0, len(texts), EMBEDDING_BATCH):
            batch = texts[i:i + EMBEDDING_BATCH]
            enc = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=MAX_SEQ,
                return_tensors='pt'
            ).to(device)
            # CLS pooling: take index-0 of last_hidden_state
            cls = model(**enc).last_hidden_state[:, 0]
            # L2 normalize (p=2)
            cls = torch.nn.functional.normalize(cls, p=2, dim=1)
            out.append(cls.cpu().numpy().astype('float32'))
    return np.vstack(out)


print('\u2705 embed_texts() defined')
print('   Pooling : CLS (index 0 of last_hidden_state)')
print('   Norm    : L2 (p=2)')
print('   Output  : float32, 1024-dim')

✅ embed_texts() defined
   Pooling : CLS (index 0 of last_hidden_state)
   Norm    : L2 (p=2)
   Output  : float32, 1024-dim


## Cell 6 — Define `point_id()` — Exact Replica of Phase 4

In [6]:
# ============================================================
# CELL 6: point_id() — exact copy of phase4/config.py lines 97-100
#
#   uuid5(NAMESPACE_URL, 'bis-sih26107-phase4/' + chunk_id)
#   Deterministic: re-running never creates duplicate points.
# ============================================================
import uuid

def point_id(chunk_id: str) -> str:
    """Exact replica of phase4/config.py point_id()."""
    return str(uuid.uuid5(uuid.NAMESPACE_URL, POINT_ID_NAMESPACE + chunk_id))


# Sanity check
_test = point_id('test-chunk-001')
assert len(_test) == 36 and _test.count('-') == 4, 'UUID format wrong'
print('\u2705 point_id() defined')
print(f'   Namespace : "{POINT_ID_NAMESPACE}"')
print(f'   Sample ID : {_test}')

✅ point_id() defined
   Namespace : "bis-sih26107-phase4/"
   Sample ID : d1c632cf-02e2-53b7-9baa-f63b0c90fe0d


## Cell 7 — Load & Validate Chunks

In [7]:
# ============================================================
# CELL 7: Load qdrant_chunks.jsonl and build payload dicts
#         Exact payload processing from
#         phase4/qdrant/embed_and_upsert.py  lines 36-50
# ============================================================
import json

print(f'Reading {FILE_PATH} ...')
chunks = []  # list of (chunk_id: str, payload: dict)

with open(FILE_PATH, encoding='utf-8') as f:
    for i, line in enumerate(f, 1):
        o = json.loads(line)
        chunk_id = (o.get('chunk_id') or '').strip()
        if not chunk_id:
            raise ValueError(f'Line {i} has no chunk_id — file may be corrupted')

        # Build payload exactly as embed_and_upsert.py does
        payload = {k: o.get(k, '') for k in PAYLOAD_FIELDS}
        for k in INT_FIELDS:
            v = str(payload.get(k) or '').strip()
            payload[k] = int(v) if v.isdigit() else None

        chunks.append((chunk_id, payload))

print(f'Loaded   : {len(chunks)} chunks')
print(f'Expected : {EXPECTED_CHUNKS} chunks')

if len(chunks) != EXPECTED_CHUNKS:
    raise ValueError(
        f'CHUNK COUNT MISMATCH: got {len(chunks)}, expected {EXPECTED_CHUNKS}.\n'
        'Re-upload qdrant_chunks.jsonl from your PC — the file may be incomplete.'
    )

# Check for duplicate chunk_ids
ids = [cid for cid, _ in chunks]
duplicates = len(ids) - len(set(ids))
if duplicates:
    raise ValueError(f'{duplicates} duplicate chunk_ids found — file corrupted')

print(f'\n\u2705 All {EXPECTED_CHUNKS} chunks loaded and validated')
print(f'   Duplicate IDs : {duplicates}')
print(f'   Sample chunk  : {chunks[0][0]} -> point {point_id(chunks[0][0])}')

Reading /content/drive/MyDrive/BIS_SIH26107/qdrant_chunks.jsonl ...
Loaded   : 45523 chunks
Expected : 45523 chunks

✅ All 45523 chunks loaded and validated
   Duplicate IDs : 0
   Sample chunk  : CHK-0000001 -> point 58cb30d5-dbe6-5f07-bb6c-b02f826be0b7


## Cell 8 — Connect Qdrant & Check Current Progress

In [8]:
# ============================================================
# CELL 8: Connect to Qdrant Cloud and read current point count.
#         This determines which shard to resume from in Cell 10.
# ============================================================
from qdrant_client import QdrantClient, models

print('Connecting to Qdrant Cloud ...')
client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY, timeout=60)

try:
    info = client.get_collection(COLLECTION)
except Exception as e:
    raise RuntimeError(
        f'Cannot reach collection "{COLLECTION}": {e}\n'
        'Check QDRANT_URL and QDRANT_API_KEY in Cell 3.'
    )

# Check vector dimension
try:
    actual_size = info.config.params.vectors.size
except Exception:
    actual_size = 'unknown'

current_count = client.count(collection_name=COLLECTION, exact=True).count
remaining     = EXPECTED_CHUNKS - current_count

print(f'\n\u2705 Connected to Qdrant Cloud')
print(f'   Collection   : {COLLECTION}')
print(f'   Vector dim   : {actual_size} (expected {VECTOR_SIZE})')
print(f'   Points now   : {current_count}')
print(f'   Expected     : {EXPECTED_CHUNKS}')
print(f'   Remaining    : {remaining}')

if actual_size not in ('unknown',) and actual_size != VECTOR_SIZE:
    raise ValueError(
        f'Collection dimension is {actual_size}, expected {VECTOR_SIZE}.\n'
        'Run phase4/qdrant/create_collection.py from your local machine first.'
    )

if remaining == 0:
    print('\n\U0001f389 Collection already complete! Run local smoke test.')

Connecting to Qdrant Cloud ...

✅ Connected to Qdrant Cloud
   Collection   : bis_knowledge
   Vector dim   : 1024 (expected 1024)
   Points now   : 2560
   Expected     : 45523
   Remaining    : 42963


## Cell 9 — Benchmark One Shard (Optional — Recommended)

In [9]:
# ============================================================
# CELL 9: Time a single 512-chunk shard to estimate total job time.
#         Optional but useful — run before committing to the full job.
# ============================================================
import time

print('Benchmarking one shard (512 chunks) ...')
test_texts = [pl['content'] for _, pl in chunks[:EMBEDDING_SHARD]]

t0   = time.time()
vecs = embed_texts(test_texts)
dt   = time.time() - t0

cps           = len(test_texts) / dt
remaining_chk = EXPECTED_CHUNKS - current_count
est_min       = (remaining_chk / cps) / 60

print(f'\n\U0001f4ca Benchmark result:')
print(f'   Chunks       : {len(test_texts)}')
print(f'   Time         : {dt:.1f} seconds')
print(f'   Throughput   : {cps:.1f} chunks/sec')
print(f'   Vector shape : {vecs.shape}')
print(f'   Vector dtype : {vecs.dtype}')
print(f'\n\u23f1\ufe0f  Estimated time for {remaining_chk} remaining chunks:')
print(f'   Embedding    : ~{est_min:.0f} min')
print(f'   With upserts : ~{est_min * 1.15:.0f} min (+15% for network)')
print('\n\u2705 Vectors look correct. Proceed to Cell 10.')

Benchmarking one shard (512 chunks) ...

📊 Benchmark result:
   Chunks       : 512
   Time         : 9.9 seconds
   Throughput   : 51.7 chunks/sec
   Vector shape : (512, 1024)
   Vector dtype : float32

⏱️  Estimated time for 42963 remaining chunks:
   Embedding    : ~14 min
   With upserts : ~16 min (+15% for network)

✅ Vectors look correct. Proceed to Cell 10.


## Cell 10 — Main Job: Embed All Shards + Upsert to Qdrant (Resumable)

In [10]:
# ============================================================
# CELL 10: Main embedding + upsert loop
#
#   RESUMABLE: reads current Qdrant count from Cell 8 and
#   skips already-completed shards automatically.
#
#   IDEMPOTENT: point IDs are UUID5 — upserting the same shard
#   twice just overwrites the same points, never duplicates.
#
#   If Colab disconnects: re-run Cells 1-8, then re-run this cell.
# ============================================================
import time

total  = len(chunks)
shards = (total + EMBEDDING_SHARD - 1) // EMBEDDING_SHARD

# Determine resume shard from current Qdrant point count
# floor division: 2560 pts -> 5 shards done -> start shard index 5
start_shard = current_count // EMBEDDING_SHARD

print(f'Total shards   : {shards}')
print(f'In Qdrant now  : {current_count} pts  ({start_shard} shards complete)')
print(f'Starting from  : shard {start_shard + 1} / {shards}')
print(f'Shards to run  : {shards - start_shard}')
print('=' * 65)

if start_shard >= shards:
    print('\u2705 All shards already in Qdrant. Nothing to do.')
else:
    upserted_this_run = 0
    t_start = time.time()

    for si in range(start_shard, shards):
        chunk_slice = chunks[si * EMBEDDING_SHARD : (si + 1) * EMBEDDING_SHARD]
        texts = [pl['content'] for _, pl in chunk_slice]

        # -- Embed --
        t0    = time.time()
        vecs  = embed_texts(texts)
        t_emb = time.time() - t0

        # -- Build Qdrant points --
        points = [
            models.PointStruct(
                id      = point_id(cid),
                vector  = v.tolist(),
                payload = pl
            )
            for (cid, pl), v in zip(chunk_slice, vecs)
        ]

        # -- Batch upsert --
        t0 = time.time()
        for j in range(0, len(points), UPSERT_BATCH):
            client.upsert(
                collection_name=COLLECTION,
                points=points[j : j + UPSERT_BATCH],
                wait=True
            )
            upserted_this_run += len(points[j : j + UPSERT_BATCH])
        t_ups = time.time() - t0

        # -- Progress line --
        elapsed     = time.time() - t_start
        done_shards = si - start_shard + 1
        left_shards = shards - si - 1
        eta_sec     = (elapsed / done_shards) * left_shards if done_shards else 0
        total_in_db = current_count + upserted_this_run

        print(
            f'Shard {si+1:3d}/{shards}  |  '
            f'{total_in_db:6d}/{EXPECTED_CHUNKS} pts  |  '
            f'emb {t_emb:5.1f}s  ups {t_ups:4.1f}s  |  '
            f'ETA ~{eta_sec/60:.0f}min',
            flush=True
        )

    elapsed_total = time.time() - t_start
    print('=' * 65)
    print(f'\u2705 Run complete.  Upserted {upserted_this_run} new points '
          f'in {elapsed_total/60:.1f} min')

Total shards   : 89
In Qdrant now  : 2560 pts  (5 shards complete)
Starting from  : shard 6 / 89
Shards to run  : 84
Shard   6/89  |    3072/45523 pts  |  emb   5.5s  ups  4.6s  |  ETA ~15min
Shard   7/89  |    3584/45523 pts  |  emb   5.0s  ups  5.7s  |  ETA ~15min
Shard   8/89  |    4096/45523 pts  |  emb   5.6s  ups  3.9s  |  ETA ~14min
Shard   9/89  |    4608/45523 pts  |  emb   5.4s  ups  3.4s  |  ETA ~13min
Shard  10/89  |    5120/45523 pts  |  emb   5.5s  ups  3.4s  |  ETA ~13min
Shard  11/89  |    5632/45523 pts  |  emb   5.5s  ups  3.3s  |  ETA ~12min
Shard  12/89  |    6144/45523 pts  |  emb   5.3s  ups  3.6s  |  ETA ~12min
Shard  13/89  |    6656/45523 pts  |  emb   5.2s  ups  3.4s  |  ETA ~12min
Shard  14/89  |    7168/45523 pts  |  emb   5.6s  ups  3.1s  |  ETA ~12min
Shard  15/89  |    7680/45523 pts  |  emb   5.8s  ups  3.3s  |  ETA ~12min
Shard  16/89  |    8192/45523 pts  |  emb   5.8s  ups  6.6s  |  ETA ~12min
Shard  17/89  |    8704/45523 pts  |  emb   5.6s  ups  3.4

## Cell 11 — Final Validation

In [11]:
# ============================================================
# CELL 11: Confirm final point count in Qdrant Cloud
# ============================================================
final_count = client.count(collection_name=COLLECTION, exact=True).count

print(f'Final Qdrant count : {final_count}')
print(f'Expected           : {EXPECTED_CHUNKS}')

if final_count == EXPECTED_CHUNKS:
    print('\n\U0001f389 SUCCESS — All 45,523 chunks are in Qdrant Cloud bis_knowledge')
    print('\nNext steps on your LOCAL PC:')
    print('  cd phase4')
    print('  python run_phase4.py --smoke-only')
    print('')
    print('  cd phase5')
    print('  python run_phase5.py --check')
elif final_count > EXPECTED_CHUNKS:
    print(f'\n\u26a0\ufe0f  OVER-COUNT: {final_count} (expected {EXPECTED_CHUNKS})')
    print('Investigate — collection may not have been freshly created.')
else:
    missing = EXPECTED_CHUNKS - final_count
    print(f'\n\u26a0\ufe0f  INCOMPLETE: {missing} chunks still missing.')
    print('Re-run Cell 8 (refresh current_count) then Cell 10 to resume.')

Final Qdrant count : 45523
Expected           : 45523

🎉 SUCCESS — All 45,523 chunks are in Qdrant Cloud bis_knowledge

Next steps on your LOCAL PC:
  cd phase4
  python run_phase4.py --smoke-only

  cd phase5
  python run_phase5.py --check
